In [0]:
import re
from pyspark.sql import functions as F

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

dbutils.widgets.text(
    "catalog",
    "training_lab_classic"
)

dbutils.widgets.text(
    "schema",
    "cold_chain"
)

dbutils.widgets.text(
    "volume",
    "volume1"
)

dbutils.widgets.text(
    "run_id",
    "manual_001"
)

dbutils.widgets.text(
    "as_of_time",
    "2026-10-09T17:00:00Z"
)

dbutils.widgets.text(
    "orchestrator",
    "manual"
)

dbutils.widgets.text(
    "airflow_dag_id",
    ""
)

dbutils.widgets.text(
    "airflow_run_id",
    ""
)

dbutils.widgets.dropdown(
    "simulate_failure",
    "false",
    ["false", "true"]
)

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")

RUN_ID = dbutils.widgets.get("run_id")
AS_OF_TIME = dbutils.widgets.get("as_of_time")

ORCHESTRATOR = dbutils.widgets.get(
    "orchestrator"
)

AIRFLOW_DAG_ID = dbutils.widgets.get(
    "airflow_dag_id"
)

AIRFLOW_RUN_ID = dbutils.widgets.get(
    "airflow_run_id"
)

SIMULATE_FAILURE = (
    dbutils.widgets
    .get("simulate_failure")
    .lower()
    == "true"
)

for name, value in {
    "catalog": CATALOG,
    "schema": SCHEMA,
    "volume": VOLUME
}.items():
    assert re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*",
        value
    ), f"Invalid {name}: {value}"

assert re.fullmatch(
    r"[A-Za-z0-9_.:-]+",
    RUN_ID
), f"Invalid run_id: {RUN_ID}"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

BASE_PATH = (
    f"/Volumes/{CATALOG}/"
    f"{SCHEMA}/{VOLUME}"
)

print("==============================================")
print("COLD CHAIN PIPELINE CONFIGURATION")
print("==============================================")

print("Namespace        :", NAMESPACE)
print("Volume path      :", BASE_PATH)
print("Run ID           :", RUN_ID)
print("Evaluation time  :", AS_OF_TIME)
print("Orchestrator     :", ORCHESTRATOR)
print("Airflow DAG ID   :", AIRFLOW_DAG_ID)
print("Airflow Run ID   :", AIRFLOW_RUN_ID)
print("Simulate failure :", SIMULATE_FAILURE)

In [0]:
AUDIT_TABLE = (
    f"{NAMESPACE}.pipeline_run_audit"
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE}
    (
        run_id STRING,
        orchestrator STRING,
        airflow_dag_id STRING,
        airflow_run_id STRING,
        databricks_job_id STRING,
        databricks_run_id STRING,
        status STRING,
        started_at TIMESTAMP,
        completed_at TIMESTAMP,
        input_rows BIGINT,
        accepted_rows BIGINT,
        rejected_rows BIGINT,
        rejected_percent DOUBLE,
        temperature_exception_count BIGINT,
        gold_temperature_window_rows BIGINT,
        shipment_health_rows BIGINT,
        delivery_performance_rows BIGINT,
        error_message STRING
    )
    USING DELTA
    """
)

print(
    "Audit table created/verified:",
    AUDIT_TABLE
)

In [0]:
assert spark.catalog.tableExists(
    AUDIT_TABLE
), "pipeline_run_audit was not created"

print("Audit table exists.")

spark.table(
    AUDIT_TABLE
).printSchema()

In [0]:
from pyspark.sql import Row
from datetime import datetime, timezone

started_at = datetime.now(
    timezone.utc
).replace(tzinfo=None)

start_record = [
    Row(
        run_id=RUN_ID,
        orchestrator=ORCHESTRATOR,
        airflow_dag_id=(
            AIRFLOW_DAG_ID
            if AIRFLOW_DAG_ID
            else None
        ),
        airflow_run_id=(
            AIRFLOW_RUN_ID
            if AIRFLOW_RUN_ID
            else None
        ),
        databricks_job_id=None,
        databricks_run_id=None,
        status="RUNNING",
        started_at=started_at,
        completed_at=None,
        input_rows=None,
        accepted_rows=None,
        rejected_rows=None,
        rejected_percent=None,
        temperature_exception_count=None,
        gold_temperature_window_rows=None,
        shipment_health_rows=None,
        delivery_performance_rows=None,
        error_message=None
    )
]

start_df = spark.createDataFrame(
    start_record,
    schema=spark.table(AUDIT_TABLE).schema
)

In [0]:
from delta.tables import DeltaTable

audit_delta = DeltaTable.forName(
    spark,
    AUDIT_TABLE
)

(
    audit_delta.alias("target")
    .merge(
        start_df.alias("source"),
        "target.run_id = source.run_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    "Run registered:",
    RUN_ID
)

In [0]:
display(
    spark.table(AUDIT_TABLE)
    .filter(
        F.col("run_id") == RUN_ID
    )
)

In [0]:
BRONZE_SENSOR_TABLE = (
    f"{NAMESPACE}.bronze_sensor_events"
)

SILVER_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_events"
)

QUARANTINE_SENSOR_TABLE = (
    f"{NAMESPACE}.quarantine_sensor_events"
)

ENRICHED_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_enriched"
)

TEMPERATURE_WINDOWS_TABLE = (
    f"{NAMESPACE}.shipment_temperature_windows"
)

SHIPMENT_HEALTH_TABLE = (
    f"{NAMESPACE}.shipment_health"
)

DELIVERY_PERFORMANCE_TABLE = (
    f"{NAMESPACE}.delivery_performance"
)

required_tables = [
    BRONZE_SENSOR_TABLE,
    SILVER_SENSOR_TABLE,
    QUARANTINE_SENSOR_TABLE,
    ENRICHED_SENSOR_TABLE,
    TEMPERATURE_WINDOWS_TABLE,
    SHIPMENT_HEALTH_TABLE,
    DELIVERY_PERFORMANCE_TABLE
]

for table_name in required_tables:
    assert spark.catalog.tableExists(
        table_name
    ), f"Missing table: {table_name}"

input_rows = spark.table(
    BRONZE_SENSOR_TABLE
).count()

accepted_rows = spark.table(
    SILVER_SENSOR_TABLE
).count()

rejected_rows = spark.table(
    QUARANTINE_SENSOR_TABLE
).count()

temperature_exception_count = (
    spark.table(
        ENRICHED_SENSOR_TABLE
    )
    .filter(
        F.col(
            "is_temperature_exception"
        ) == True
    )
    .count()
)

gold_temperature_window_rows = (
    spark.table(
        TEMPERATURE_WINDOWS_TABLE
    )
    .count()
)

shipment_health_rows = (
    spark.table(
        SHIPMENT_HEALTH_TABLE
    )
    .count()
)

delivery_performance_rows = (
    spark.table(
        DELIVERY_PERFORMANCE_TABLE
    )
    .count()
)

if input_rows > 0:
    rejected_percent = (
        rejected_rows
        / input_rows
    ) * 100.0
else:
    rejected_percent = None


In [0]:
print("==============================================")
print("PIPELINE METRICS")
print("==============================================")

print("Input rows                  :", input_rows)
print("Accepted rows               :", accepted_rows)
print("Rejected rows               :", rejected_rows)
print("Rejected percent            :", rejected_percent)

print(
    "Temperature exceptions      :",
    temperature_exception_count
)

print(
    "Gold temperature windows    :",
    gold_temperature_window_rows
)

print(
    "Shipment health rows        :",
    shipment_health_rows
)

print(
    "Delivery performance rows   :",
    delivery_performance_rows
)

In [0]:
completed_at = datetime.now(
    timezone.utc
).replace(tzinfo=None)

success_record = [
    Row(
        run_id=RUN_ID,
        orchestrator=ORCHESTRATOR,
        airflow_dag_id=(
            AIRFLOW_DAG_ID
            if AIRFLOW_DAG_ID
            else None
        ),
        airflow_run_id=(
            AIRFLOW_RUN_ID
            if AIRFLOW_RUN_ID
            else None
        ),
        databricks_job_id=None,
        databricks_run_id=None    )
]

